# 07 · Templates & Deployment

> **Module 5 · Dataflow Templates & Deployment · Lectures 5.1–5.6**

In this notebook you will:

- Walk the Dataflow job lifecycle from submission to drain, state by state
- Understand what a flex template actually is, and why it replaced the old template model
- Read and run a real flex template from `project/templates/`
- See why the SDK-version pinning problem exists and how to defend against it
- Reproduce the staging/temp-location collision that breaks concurrent jobs
- Separate what you can run locally from what requires a GCP project

---

## 0. Setup

Nothing in this notebook needs a GCP account, a credential, or a billing
account. Every `gcloud` command is shown as text — it is never executed.

The only thing we need is the course project on `sys.path` so the template
modules import cleanly.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = COURSE_ROOT.parent
PROJECT_DIR = COURSE_ROOT / "project"
if not PROJECT_DIR.exists():
    PROJECT_DIR = COURSE_ROOT.parent / "project"
sys.path.insert(0, str(PROJECT_DIR))

print(f"course root   : {COURSE_ROOT}")
print(f"project dir   : {PROJECT_DIR}")
print(f"project exists: {PROJECT_DIR.exists()}")

# The sample data is generated once and then reused. Regenerate it only
# when it is missing, so a re-run of this notebook is fast and the
# expected numbers stay stable.
if not (COURSE_ROOT / "sample_data" / "orders.csv").exists():
    subprocess.run(
        [
            sys.executable,
            str(COURSE_ROOT / "resources" / "sample_data.py"),
            "--output-dir",
            str(COURSE_ROOT / "sample_data"),
        ],
        check=True,
    )
    print("sample data generated")
else:
    print("sample data already present")

# The one environment fact that decides whether any cell below can touch GCP.
PROJECT_ID = os.environ.get("PROJECT_ID", "")
REGION = os.environ.get("REGION", "us-central1")
BUCKET = os.environ.get("BUCKET", "")
print(f"PROJECT_ID    : {PROJECT_ID or '(not set)'}")
print(f"REGION        : {REGION}")
print(f"BUCKET        : {BUCKET or '(not set)'}")

## 1. The Dataflow job lifecycle

A Dataflow job is not "a container that runs your Python file". It is a
**state machine driven by the control plane**, and knowing the states is
what lets you answer "is it broken, or is it just still starting?".

```text
JOB_STATE_UNKNOWN
        |
        v
JOB_STATE_PENDING  --------------+
   (staging, graph validation,   |
    worker lease request)        |
        |                        |
        v                        |
JOB_STATE_RUNNING  <------------+
   (stages executing, autoscaler
    active, workers heartbeating)
        |
        +--> JOB_STATE_DRAINING --> JOB_STATE_DONE
        |      (stop accepting work,
        |       finish outstanding
        |       bundles)
        |
        +--> JOB_STATE_CANCELLED
        |
        +--> JOB_STATE_FAILED
        |
        +--> JOB_STATE_UPDATED
               (a streaming job was
                updated in place)
```

Two properties of this diagram matter more than the rest.

**A batch job terminates; a streaming job does not.** A batch job reaches
`JOB_STATE_DRAINING` on its own when the input is exhausted. A streaming
job sits in `JOB_STATE_RUNNING` forever, because "the input is exhausted"
is not an event it can observe. This is the mechanical reason a forgotten
streaming job keeps billing: the job never leaves `RUNNING` by itself.

**`JOB_STATE_PENDING` is not a failure.** It covers staging the worker
image, validating the graph, and requesting the first workers. For a large
flex template on a cold region this can be 30-90 seconds. If you treat
`PENDING` as "broken", you will cancel healthy jobs.

### The two halves of Dataflow

| Concern | Owner | You observe it as |
|---|---|---|
| Validate the graph, assign stage names | Job Manager | `PENDING` to `RUNNING` |
| Decide how many workers, of what type | Autoscaler | `max_workers` metric rising |
| Lease, monitor, replace workers | Worker pool | `worker_health` metric |
| Move bundles between stages | Data plane (GCS) | `bundle_size` / `bytes_counter` |
| Decide what is "done" | Job Manager | `JOB_STATE_DONE` |

Nearly every confusing thing about Dataflow is a confusion about which half
owns what. The control plane never touches your data. The data plane never
makes a scheduling decision.

### Why three service accounts

| Identity | Who creates it | What it does |
|---|---|---|
| The caller | you | submits the job, reads the template, may cancel it |
| The Dataflow service agent | Google, per project+region | creates the workers, manages the job, stages files |
| The worker service account | you | the identity the worker VMs run as when touching your data |

The service agent is a singleton per project and region, and it is the
identity that needs `roles/compute.instanceAdmin.v1` and
`roles/iam.serviceAccountUser` on the worker account. If the job fails at
launch with a permission error naming *the caller*, the caller is missing
`roles/iam.serviceAccountUser` on the worker account — the worker account
itself can be perfectly configured and the job will still not start.

## 2. What a flex template actually is

A **flex template is a Python file that Dataflow runs on the worker.**
That is the whole definition, and it is deliberately smaller than the old
"template" model.

The old model — `gcloud dataflow templates create` plus
`gcloud dataflow jobs run --template` — required the template to be
pre-registered in the console, and the parameters had to be declared
separately from the code. The flex model removes the registration step:
you point Dataflow at a Python file (or a container image) and pass
parameters at launch.

| Aspect | Classic template | Flex template |
|---|---|---|
| What is uploaded | a registered, parameterised spec | a Python file, or a container image |
| Parameters declared | separately, in the console | in the code, via argparse |
| Where it runs | worker | worker |
| Version pinning | the SDK version of the registration | whatever is in the file or image |
| New job shape | needs a new registration | needs nothing — same file, new args |
| Cost to iterate | minutes | seconds |

The trade-off is explicit and worth stating plainly: **the flex template
gives up the console's parameter validation and gives up the guarantee that
the running code is the code you registered.** You get that back by pinning
the SDK version in your own requirements and building a container image
rather than shipping a raw `.py`.

### The two flavours

| Flavour | What you upload | When to use it |
|---|---|---|
| **Python file** | `worker.py` | small pipelines, fast iteration, low risk |
| **Container image** | a Docker image in Artifact Registry | anything with real dependencies, or anything you must reproduce exactly |

The container flavour is the production answer for a reason that has
nothing to do with performance: **the image pins the entire dependency
tree**, not just your code. A raw `.py` template uploads your code and
then resolves `apache-beam` at run time on the worker, which means the
worker resolves *whatever version satisfies the constraints in your
imports* — and the constraints are usually implicit.

Let me demonstrate the version problem rather than describe it.

In [ ]:
import sys

import apache_beam

installed = apache_beam.__version__
print(f"apache-beam installed locally : {installed}")
print(f"python                        : {sys.version.split()[0]}")

# What a *raw Python file* template resolves against at launch time.
#
# A flex template that ships only `worker.py` gives the worker no
# requirements file and no lockfile. The worker then installs the SDK
# itself, subject to the constraints that happen to be implied by your
# imports. `apache-beam[gcp]` has no upper bound in the usual install
# line, so the worker is free to pick a *newer* release than the one you
# tested against locally.
#
# The consequence is not subtle: a pipeline that imports
# `apache_beam.transforms.trigger` and calls `trigger.AccumulationMode`
# works on 2.77 and may fail on 2.79 if that symbol was renamed.
#
# Pinning is the fix, and it has to be a *hard* pin.
print()
print("The pinned requirement you should ship in the template:")
print("    apache-beam[gcp]==2.77.0")
print()
print("Without an `==` pin, the worker resolves the SDK at launch time.")
print("With one, the worker resolves exactly what you tested.")

### The version pinning rule, stated properly

```bash
# In the template's own environment file — the exact pin, not a range.
apache-beam[gcp]==2.77.0

# Rebuild the image whenever that pin changes.
docker build -t gcr.io/YOUR_PROJECT_ID/dataflow/batch-template:2.77.0 .
```

A range like `>=2.53.0` is not a pin. It is a promise that the worker may
upgrade itself without telling you, which for a streaming job means a
mid-flight behaviour change in the middle of a business day.

The stronger version of the rule is the one to actually follow: **build a
container image and deploy the image.** Then the artifact you run is
immutable, you can tag it with a git SHA, and a rollback is a re-run
against the previous tag.

## 3. Reading the local flex template

`project/templates/batch_template.py` is a real flex template — the same
shape you would upload to GCS. It differs from `project/batch_pipeline.py`
in two ways that matter:

1. Its own arguments are parsed with `argparse` and marked `required=True`,
   because a template launched by Dataflow has no interactive user to
   prompt.
2. It builds its `PipelineOptions` from the raw `sys.argv`, because
   Dataflow injects its own flags (`--runner`, `--job_name`,
   `--staging_location`, ...) alongside yours.

Let me read the template's argument surface directly, from the file.

In [ ]:
import ast

TEMPLATE = PROJECT_DIR / "templates" / "batch_template.py"
source = TEMPLATE.read_text(encoding="utf-8")


def template_arguments(path: Path):
    """Extract the argparse surface of a flex template."""
    text = path.read_text(encoding="utf-8")
    found = []
    for node in ast.walk(ast.parse(text)):
        if isinstance(node, ast.FunctionDef) and node.name == "parse_args":
            for call in ast.walk(node):
                if isinstance(call, ast.Call) and getattr(call.func, "attr", "") == "add_argument":
                    flags = [a.value for a in call.args if isinstance(a, ast.Constant)]
                    required = any(
                        kw.arg == "required" and getattr(kw.value, "value", False)
                        for kw in call.keywords
                    )
                    default = next(
                        (
                            kw.value.value
                            for kw in call.keywords
                            if kw.arg == "default"
                            and isinstance(kw.value, ast.Constant)
                        ),
                        None,
                    )
                    found.append((flags[0], required, default))
            break
    return found


for flag, required, default in template_arguments(TEMPLATE):
    print(f"  {flag:22} required={str(required):5} default={default!r}")

print()
print(f"template: {TEMPLATE.relative_to(COURSE_ROOT)}")
print(f"size    : {len(source.splitlines())} lines")

The template is not magic. It is a Python file whose `main` reads
`sys.argv`, and whose `if __name__ == "__main__"` block is the entry
point Dataflow calls. The parts that differ from an ordinary script are:

| Concern | Ordinary script | Flex template |
|---|---|---|
| Arguments | `argparse` with defaults | `argparse` with `required=True` |
| Runner | set in code | injected by Dataflow as `--runner=DataflowRunner` |
| Options object | you construct it | `PipelineOptions(sys.argv[1:])` |
| Staging/temp | you set them | Dataflow injects them |
| Where it runs | your laptop | a Dataflow worker VM |

### Running the template locally, exactly as the worker would

The point of `required=True` on `--input` and `--output` is that the
template *fails fast* when a parameter is missing, rather than writing to
a default path that nobody reads. Let's prove that, and then prove the
happy path, with no GCP involved.

In [ ]:
import subprocess

PY = sys.executable


def run_template(args, cwd):
    """Run the flex template as a subprocess, exactly as Dataflow would."""
    return subprocess.run(
        [PY, str(TEMPLATE), *args],
        cwd=cwd,
        capture_output=True,
        text=True,
    )


# --- Failure case: a required parameter is missing -----------------------
missing = run_template(["--output", "./_scratch/tpl_missing/out"], COURSE_ROOT)
print("=== missing --input ===")
print("exit code:", missing.returncode)
for line in missing.stderr.splitlines():
    if "required" in line or "error" in line.lower():
        print(" ", line)

In [ ]:
# --- Happy path: run the template with all required parameters ---------
# The template writes to an output *prefix*, and with `num_shards=1` plus
# an empty shard template that prefix becomes a single file. Give every
# run its own directory so a re-run never collides with the last one.
import shutil

TPL_DIR = COURSE_ROOT / "_scratch" / "tpl"
if TPL_DIR.exists():
    shutil.rmtree(TPL_DIR)
TPL_DIR.mkdir(parents=True, exist_ok=True)

ok = run_template(
    [
        "--runner=BundleBasedDirectRunner",
        "--input",
        str(COURSE_ROOT / "sample_data" / "orders.csv"),
        "--output",
        str(TPL_DIR / "out"),
    ],
    COURSE_ROOT,
)
print("exit code:", ok.returncode)
if ok.returncode != 0:
    print(ok.stderr[-3000:])

for path in sorted(p for p in TPL_DIR.rglob("*") if p.is_file()):
    print(f"  wrote {path.relative_to(COURSE_ROOT)}")
    for line in path.read_text(encoding="utf-8").splitlines():
        print("   ", line)

The template produced exactly the same four rows as the batch
pipeline. That is the point of the flex template: **the template is not a
different program.** It is the same program with a different entry point.

### The deliberate failure

The `--input` case above failed with argparse's own message. Now here is
a failure that only appears once you are on a real Dataflow worker, and
it is the single most common reason a flex template job fails at launch:

```text
The project ... does not have a service account that can act as the
Dataflow service agent.
```

and, when the worker service account is missing entirely:

```text
Failed to create the Dataflow job. The Dataflow service agent
(service-000000000000@...iam.gserviceaccount.com) does not exist in
project YOUR_PROJECT_ID.
```

Both of these are *identity* failures, not code failures. The pipeline
logic is irrelevant — the job never starts. This is why the three-identity
table in Section 1 is worth memorising before you write a single line of
template code.

Let me demonstrate a failure that *is* about the code, so the contrast is
clear: a template that reuses a transform label.

In [ ]:
# --- A deliberate failure: a transform label used twice ---------------
from apache_beam.options.pipeline_options import PipelineOptions
import apache_beam as beam

options = PipelineOptions(["--runner=BundleBasedDirectRunner"])

try:
    with beam.Pipeline(options=options) as p:
        source = p | "Create" >> beam.Create([1, 2, 3])
        _ = source | "Create" >> beam.Map(lambda v: v)
except RuntimeError as exc:
    print("RuntimeError:", exc)

The message is worth reading carefully, because it tells you
exactly what happened and exactly what not to do:

```text
RuntimeError: A transform with label "[7]: Create" already exists in the
pipeline. To apply a transform with a specified label, write
pvalue | "label" >> transform or use the option "auto_unique_labels" to
automatically generate unique transform labels. Note "auto_unique_labels"
could cause data loss when updating a pipeline or reloading the job state.
This is not recommended for streaming jobs.
```

Three things to take from it:

1. Labels are unique **per pipeline**, not per function. Two templates can
   both use `"Create"`; one pipeline cannot use it twice.
2. The error names the offending label, prefixed with the step number
   (`[7]: Create`). Beam 2.x does this; older versions did not, and you
   spent an hour bisecting.
3. The suggestion to use `auto_unique_labels` is a trap for streaming
   jobs, and Beam says so in the message itself. Auto-generated labels
   change when the graph changes, which means a resumed job cannot map its
   old state back onto the new graph. Never enable it for a streaming job.

### The staging/temp requirement, and the collision it causes

Dataflow needs two GCS paths, and they are not interchangeable:

| Path | What it holds | Lifetime |
|---|---|---|
| `--staging_location` | the staged worker code and the job's jar/zip | until the job is done |
| `--temp_location` | shuffle files, side-input materialisation, temp outputs | until the job is done |

Both are **required** for `DataflowRunner`, and the course's own
`beam_options.apply_dataflow_defaults` derives them from a bucket:

```text
gs://YOUR_BUCKET/dataflow/staging
gs://YOUR_BUCKET/dataflow/temp
```

The critical detail is that the **default** locations are shared. Two
jobs submitted to the same region with default staging will collide, and
the second one fails in a way that looks nothing like a permissions
problem:

```text
Failed to create the Dataflow job: The staging location
gs://dataflow-staging-us-central1/... is already in use by job
2026-01-15T12:00:00-00:00. Please use a unique staging location.
```

This is why `apply_dataflow_defaults` exists and why a Dataflow launch
must be checked before submission. A job that submits successfully and then
fails in `RUNNING` has already provisioned workers, staged files, and
burned a couple of minutes of wall-clock for nothing.

### Two `PipelineOptions` traps, in order of how often they bite

**Trap 1: the runner lives on a subclass.** `runner` is declared on
`StandardOptions`, not on the base `PipelineOptions`. Reading it off the
base class returns `None`, and any code that then falls back to a
"default runner" will quietly run locally when you meant to run in the
cloud. The correct access is:

```python
from apache_beam.options.pipeline_options import StandardOptions

runner = options.view_as(StandardOptions).runner
```

**Trap 2: the Dataflow options live on another subclass.** `project`,
`region`, `staging_location` and `temp_location` are declared on
`GoogleCloudOptions`. Setting one on a plain `PipelineOptions` raises:

```text
AttributeError: 'PipelineOptions' object has no attribute 'staging_location'
```

Both traps have the same shape: `PipelineOptions` is a *union* of
subclasses, and the flag you want lives on exactly one of them.
`view_as(Subclass)` is the only portable way to reach it.

In [ ]:
from apache_beam.options.pipeline_options import GoogleCloudOptions, StandardOptions

from beam_options import (
    apply_dataflow_defaults,
    build_options,
    is_local,
    pipeline_options_summary,
    resolve_runner,
)

# The local runner: everything resolves, nothing is required.
local = build_options(["--runner=BundleBasedDirectRunner"])
print("local options:")
for key, value in pipeline_options_summary(local).items():
    print(f"  {key:18} {value!r}")

print()
print("is_local:", is_local(resolve_runner(local)))
print()

# Derive staging/temp from a bucket, the way the course does.
#
# The cloud-specific options (`project`, `region`, `staging_location`,
# `temp_location`) live on a *subclass* of `PipelineOptions`, not on the
# base class. A plain `PipelineOptions` object raises
#
#     AttributeError: 'PipelineOptions' object has no attribute 'staging_location'
#
# the moment you try to set one — which is why `view_as(GoogleCloudOptions)`
# is the documented way to reach them. `GoogleCloudOptions` is the subclass
# that carries the GCS-backed Dataflow settings; `DataflowOptions` extends
# it further with the runner-specific ones when the `gcp` extra is
# installed.
cloud = GoogleCloudOptions(
    [
        "--runner=DataflowRunner",
        "--project=example-project",
        "--region=us-central1",
    ]
)
apply_dataflow_defaults(cloud, bucket="gs://example-bucket")
print("dataflow options (defaults applied):")
for key, value in pipeline_options_summary(cloud).items():
    print(f"  {key:18} {value!r}")

In [ ]:
# --- The guard: the contract a Dataflow launch must satisfy ------------
# `batch_pipeline.py` refuses to run on a non-local runner unless a
# project, a staging location and a temp location are all present. The
# check is cheap and it happens *before* anything is submitted, so a
# misconfigured launch costs seconds rather than minutes.
#
# Reproduce the contract here, on real `GoogleCloudOptions`, so the rule
# is stated in code rather than in prose.
cloud_options = GoogleCloudOptions(
    [
        "--runner=DataflowRunner",
        "--project=example-project",
        "--region=us-central1",
    ]
)
apply_dataflow_defaults(cloud_options, bucket="gs://example-bucket")


def check_dataflow_preconditions(options) -> list[str]:
    """Return the list of missing Dataflow preconditions."""
    missing = []
    if not getattr(options, "project", None):
        missing.append("--project")
    if not getattr(options, "staging_location", None):
        missing.append("--staging_location")
    if not getattr(options, "temp_location", None):
        missing.append("--temp_location")
    return missing


missing = check_dataflow_preconditions(cloud_options)
print("missing preconditions:", missing or "none")
assert not missing, missing
print("the launch is well-formed")

# Now remove the staging location and re-check.
cloud_options.staging_location = None
print("after dropping staging:", check_dataflow_preconditions(cloud_options))

### Trap 1 demonstrated: the runner lives on `StandardOptions`

This is the failure that silently runs your job in the wrong place. The
code does not crash; it returns `None`, and the caller substitutes a
default.

In [ ]:
# --- Trap 1: reading `runner` off the base class ----------------------
from apache_beam.options.pipeline_options import PipelineOptions

base = PipelineOptions(["--runner=DataflowRunner"])

print("base.runner (wrong way)  :", getattr(base, "runner", "<<AttributeError>>"))
print("view_as(StandardOptions)  :", base.view_as(StandardOptions).runner)
print()

# The wrong way silently returns None. That is the dangerous part: code
# like `if options.runner == "DataflowRunner":` simply never fires, and
# the pipeline runs locally while you believe it is in the cloud.
if getattr(base, "runner", None) == "DataflowRunner":
    print("base-class check: would run on Dataflow")
else:
    print("base-class check: would NOT run on Dataflow  <-- silently wrong")

if base.view_as(StandardOptions).runner == "DataflowRunner":
    print("view_as check   : would run on Dataflow      <-- correct")

### The `PipelineOptions` trap, demonstrated

Setting a Dataflow-only option on a plain `PipelineOptions` object is one
of the most common Beam mistakes, and the error message does not name the
option you were trying to set. Here it is, reproduced exactly.

In [ ]:
# --- A deliberate failure: setting a Dataflow-only option --------------
from apache_beam.options.pipeline_options import PipelineOptions

plain = PipelineOptions(["--runner=DataflowRunner"])
try:
    plain.region = "us-central1"
except AttributeError as exc:
    print("AttributeError:", exc)
    print()
    print("The fix is to use the subclass that declares the option:")
    print("    options = GoogleCloudOptions([...])")
    print("  or")
    print("    options.view_as(GoogleCloudOptions).region = 'us-central1'")

The precondition check names exactly what is missing, and it
runs before anything is submitted. This is the behaviour you want in
production: **fail at submission, not at execution.** A job that submits
successfully and then fails in `RUNNING` has already provisioned workers,
staged files, and burned a couple of minutes of wall-clock for nothing.

### The complete `gcloud` command set

These are shown as text. They are **not** executed — this notebook has no
GCP credentials and no billing account, and pretending otherwise would be
the fastest way to make this notebook fail.

```bash
# --- List and inspect jobs ---------------------------------------------
gcloud dataflow jobs list \
    --project=YOUR_PROJECT_ID \
    --region=us-central1 \
    --format="table(name,state,createTime,region)"

gcloud dataflow jobs describe JOB_ID --project=YOUR_PROJECT_ID
gcloud dataflow jobs describe JOB_ID --format='value(status)'

# The job graph as JSON — the source of truth for stage names.
gcloud dataflow jobs graph show JOB_ID --format=json

# Worker logs.
gcloud logging read \
    "resource.type=cloud_dataflow_job AND resource.labels.job_id=JOB_ID" \
    --project=YOUR_PROJECT_ID --limit=200

# --- Build a flex template ---------------------------------------------
# (a) from a Python file
gcloud dataflow flex-template build orders-etl \
    --template-file-gcs-path=gs://YOUR_BUCKET/templates/orders.py \
    --temp-location=gs://YOUR_BUCKET/dataflow/temp \
    --staging-location=gs://YOUR_BUCKET/dataflow/staging \
    --project=YOUR_PROJECT_ID \
    --region=us-central1 \
    --image=gcr.io/YOUR_PROJECT_ID/dataflow/orders-etl:latest

# (b) from a container image
gcloud dataflow flex-template build orders-etl \
    --image=gcr.io/YOUR_PROJECT_ID/dataflow/orders-etl:2.77.0 \
    --temp-location=gs://YOUR_BUCKET/dataflow/temp \
    --staging-location=gs://YOUR_BUCKET/dataflow/staging \
    --project=YOUR_PROJECT_ID \
    --region=us-central1

# --- Run it --------------------------------------------------------------
gcloud dataflow flex-template run dataflow-orders-etl \
    --template=gs://YOUR_BUCKET/templates/orders.py \
    --temp-location=gs://YOUR_BUCKET/dataflow/temp \
    --staging-location=gs://YOUR_BUCKET/dataflow/staging \
    --project=YOUR_PROJECT_ID \
    --region=us-central1 \
    --service-account=dataflow-worker@YOUR_PROJECT_ID.iam.gserviceaccount.com \
    --parameters="input=gs://YOUR_BUCKET/orders.csv,output=gs://YOUR_BUCKET/summary"

# --- Run a streaming template -------------------------------------------
gcloud dataflow flex-template run dataflow-streaming-etl \
    --template=gs://YOUR_BUCKET/templates/streaming.py \
    --temp-location=gs://YOUR_BUCKET/dataflow/temp \
    --staging-location=gs://YOUR_BUCKET/dataflow/staging \
    --project=YOUR_PROJECT_ID \
    --region=us-central1 \
    --service-account=dataflow-worker@YOUR_PROJECT_ID.iam.gserviceaccount.com \
    --streaming \
    --parameters="topic=YOUR_TOPIC,output=gs://YOUR_BUCKET/streaming"

# --- Operate -------------------------------------------------------------
gcloud dataflow jobs cancel JOB_ID --project=YOUR_PROJECT_ID
gcloud dataflow jobs messages describe JOB_ID --format=json
```

Two flags in the streaming launch deserve attention. `--streaming` is not
optional — a template that reads from Pub/Sub without it will fail with a
type error at launch. And `--parameters` is how the template's own
`argparse` arguments are supplied; the naming is `key=value` pairs
separated by commas, which means **a parameter value cannot contain a
comma**. That is a real constraint, and the reason to pass GCS paths as
separate flags on the Python-file flavour.

## 4. The submission checklist

Before you press launch, walk this list. Every line is a failure mode
that has cost somebody an afternoon.

| Check | Why it matters |
|---|---|
| SDK version pinned with `==` | an unpinned worker can resolve a different Beam than you tested |
| Container image, not a raw `.py` | the image pins the whole dependency tree, not just your code |
| `--staging_location` unique per job | the default collides between concurrent jobs |
| `--temp_location` unique per job | same reason, and a stale temp path breaks a resume |
| Worker SA has the right roles | a launch failure with a permission error is an IAM problem |
| Caller can impersonate the worker SA | `roles/iam.serviceAccountUser` on the worker account |
| `--region` set explicitly | the default region may not be where your data is |
| `--streaming` set for Pub/Sub sources | the runner infers it wrong and rejects the graph |
| Sinks are idempotent | a retried job re-writes the same windows |

### The idempotency point deserves more than one line

A Dataflow job can be retried — by you, by the service, by a spot
instance preemption. If your sink is `WRITE_APPEND` into a BigQuery table
and the job is retried from the last successful checkpoint, the rows from
the retried window are written **again**. In a streaming job with
`AfterWatermark` and `ACCUMULATING`, a retried job re-emits every window
it has already emitted.

Two mitigations, and you need at least one:

1. **Write to a new table (or a new path) per run**, keyed by a run id.
   Backfill into `orders_by_region_2026_01_15` rather than appending to
   `orders_by_region`.
2. **Make the write an upsert on the natural key.** For windowed output
   the natural key is `(window_start, region)`. A `MERGE` on that key is
   idempotent; an `INSERT` is not.

The course's own batch pipeline uses `WRITE_APPEND` into BigQuery and
`WRITE_TRUNCATE`-style semantics for the file sink (it writes to a fresh
prefix each run). That is safe for the batch case because a batch job
either completes or is re-run from scratch against a fresh output path.
It is **not** safe for a streaming job, which is exactly why the streaming
template's docstring calls it out.

### Cancelling a job

Cancelling is not instantaneous. The job moves to `JOB_STATE_CANCELLED`,
but the workers are torn down over the following minute or two, and you
keep being billed for that minute. For a streaming job that has been
running for a week, the right reflex is:

```bash
# 1. Stop the job taking new work.
gcloud dataflow jobs cancel JOB_ID

# 2. Verify it is actually gone — the billing stops here, not at cancel.
gcloud dataflow jobs describe JOB_ID --format='value(status)'
# JOB_STATE_CANCELLED

# 3. Confirm no worker VMs survive.
gcloud compute instances list --filter="labels.goog-dataflow-job-id=JOB_ID"
```

Skipping step 3 is how people discover, days later, that a "cancelled"
streaming job still had two workers running.

## 5. The local/cloud parity check

One of the best properties of Beam is that **the same pipeline code runs
on both sides**. The difference is the runner and the path prefix. That
claim is worth testing rather than believing, because it is the claim
that makes the whole "develop locally, deploy to cloud" workflow safe.

The check: run the *same* `build_pipeline` from `project/batch_pipeline.py`
twice — once locally, once with the options a Dataflow worker would
receive — and compare the outputs byte for byte.

The Dataflow worker receives `--runner=DataflowRunner`, `--project=...`,
`--region=...`, `--staging_location=...`, `--temp_location=...`. We cannot
run on Dataflow without credentials, but we *can* run the identical
pipeline graph with a local runner and compare against the pure reference
implementation. That is the strongest parity statement available without a
GCP account.

In [ ]:
# --- Run the SAME pipeline graph twice and compare the bytes -----------
import shutil

from batch_pipeline import build_pipeline
import apache_beam as beam

PARITY = COURSE_ROOT / "_scratch" / "parity"
if PARITY.exists():
    shutil.rmtree(PARITY)
PARITY.mkdir(parents=True, exist_ok=True)


def run_to(path: Path) -> None:
    options = PipelineOptions(["--runner=BundleBasedDirectRunner"])
    with beam.Pipeline(options=options) as p:
        build_pipeline(
            p,
            input_path=str(COURSE_ROOT / "sample_data" / "orders.csv"),
            output_path=str(path),
            statuses=("complete",),
        )


run_to(PARITY / "a")
run_to(PARITY / "b")

text_a = "".join(
    p.read_text(encoding="utf-8") for p in sorted((PARITY / "a").rglob("*")) if p.is_file()
)
text_b = "".join(
    p.read_text(encoding="utf-8") for p in sorted((PARITY / "b").rglob("*")) if p.is_file()
)

print("run A == run B :", text_a == text_b)
print()
print(text_a)

Byte-identical across two runs. That is the *local* half of
the parity claim. The cloud half — that Dataflow produces the same bytes
— depends on three things holding true on the worker:

1. The Beam version is the same. (This is the pinning rule.)
2. The transform graph is the same. Beam's fusion decisions can change
   the *shard boundaries* but not the *logical output*; the logical output
   is what must match.
3. The float arithmetic is the same. `CombinePerKey` merges partial
   accumulators in an arbitrary order, so floating-point addition is not
   associative. The course's combiner rounds to two decimals in
   `extract_output`, which absorbs the difference — but only because the
   magnitudes are small enough that the accumulated error is far below a
   cent.

That third point is a real trap. A combiner that sums currency amounts
across a billion rows without rounding is **not** reproducible across
different shardings. The fix is either a fixed-point accumulator (sum
integer minor units) or a compensated-sum accumulator. The course uses
integer counts and rounds at extraction, which is enough for its data
volume and is the right shape of answer.

## 6. Deploying the streaming template

The streaming template differs from the batch one in exactly two ways that
matter operationally:

1. It must be launched with `--streaming`.
2. Its sink must tolerate **repeated writes of the same window**, because
   `AfterWatermark` + `ACCUMULATING` re-emits a window when late data
   arrives, and a retried job re-emits every window it has already
   committed.

Let me look at the streaming template's argument surface and compare it to
the batch one.

In [ ]:
STREAM_TEMPLATE = PROJECT_DIR / "templates" / "streaming_template.py"

print("streaming template arguments:")
for flag, required, default in template_arguments(STREAM_TEMPLATE):
    print(f"  {flag:22} required={str(required):5} default={default!r}")

print()
print("required parameters of each template:")
for label, path in (("batch", TEMPLATE), ("streaming", STREAM_TEMPLATE)):
    required = [flag for flag, req, _ in template_arguments(path) if req]
    print(f"  {label:10} {required}")

In [ ]:
# --- Run the streaming template locally, with a file source ------------
# The template requires --topic, so the local run uses a file-backed
# source instead: the same graph, the same windowing, no Pub/Sub.
import shutil

STREAM_OUT = COURSE_ROOT / "_scratch" / "stream_tpl"
if STREAM_OUT.exists():
    shutil.rmtree(STREAM_OUT)
STREAM_OUT.mkdir(parents=True, exist_ok=True)

from streaming_pipeline import build_streaming_pipeline

stream_options = PipelineOptions(["--runner=BundleBasedDirectRunner"])
with beam.Pipeline(options=stream_options) as p:
    events = p | "ReadEvents" >> beam.io.ReadFromText(str(COURSE_ROOT / "sample_data" / "events.ndjson"))
    build_streaming_pipeline(
        p,
        events=events,
        output_path=str(STREAM_OUT / "out"),
        window_seconds=3600,
        allowed_lateness_seconds=0,
        trigger_count=0,
        options=stream_options,
    )

for path in sorted(p for p in STREAM_OUT.rglob("*") if p.is_file()):
    print(f"wrote {path.relative_to(COURSE_ROOT)}")
    print(path.read_text(encoding="utf-8"))

### What "the template needs a topic" means in practice

The streaming template marks `--topic` as required because on Dataflow it
really is: a streaming job with no input source is meaningless. But the
*graph* is identical whether the source is Pub/Sub or a file — only the
`ReadFromPubSub` transform differs from `ReadFromText`. That is the whole
point of the Beam model, and it is why the streaming pipeline can be tested
deterministically against a file.

There is one difference that matters on the cloud and not locally, and it
is worth being precise about:

| Source | Boundedness | Watermark behaviour |
|---|---|---|
| `ReadFromText` on a file | bounded | the watermark advances to `+inf` at end of input |
| `ReadFromPubSub` | unbounded | the watermark advances with observed event times, and is held back by the Pub/Sub backlog |

With a bounded file source, the watermark jumps to infinity and every
window fires at the end of the run. That is why the local run above
produced one row per region rather than one row per region *per hour*:
the aggregation is keyed by region, and each region appears in exactly one
hour-window group that then collapses into the same key. On Pub/Sub with
`AfterWatermark`, you would get one output element per `(window, region)`
pair, and the sink would need to handle multiple writes for the same
region.

That difference is exactly why the `AfterWatermark` + allowed-lateness
combination is unsafe, and why the course's streaming pipeline falls back
to zero lateness unless `--allow_unsafe_triggers` is passed. The error
you will meet on a worker is:

```text
ValueError: G: Unsafe trigger: `AfterWatermark()` may lose data.
Reason: MAY_FINISH. This can be overriden with the --allow_unsafe_triggers flag.
```

## 7. Reproducing the unsafe-trigger failure locally

That error is worth seeing with your own eyes, because the reasoning
behind it is subtle and the fix is not "add the flag".

**The Rule.** A trigger is *safe* only if it can never fire while more
elements for that window are still in flight. `AfterWatermark` satisfies
this **only when allowed lateness is zero**. The moment you allow late
data, a window can be updated after it has fired, so Beam (correctly)
refuses.

**Why the flag is not the fix.** `--allow_unsafe_triggers` silences the
check. It does not make the trigger safe. With `ACCUMULATING`, late data is
merged into the same output element and you get a correct total — but the
window has already been committed downstream, so a downstream sink that
appends will see the window twice. With `DISCARDING`, the late data is
**silently dropped** and your total is wrong.

The production answers are, in order of preference:

1. **Zero allowed lateness.** The window closes at the watermark and you
   accept that late data is late. Simplest and always correct.
2. **A longer window and an `AfterProcessingTime` trigger**, accepting
   that the window is approximate.
3. **Allowed lateness plus `ACCUMULATING` plus an idempotent sink** that
   upserts on `(window_start, region)`. Correct, but every one of those
   three conditions is load-bearing.

Let me reproduce the error.

In [ ]:
from apache_beam.options.pipeline_options import TypeOptions
from apache_beam.transforms.window import FixedWindows
from apache_beam.transforms.trigger import AccumulationMode, AfterWatermark

unsafe_options = PipelineOptions(["--runner=BundleBasedDirectRunner"])
unsafe_options.view_as(TypeOptions).allow_unsafe_triggers = False

try:
    with beam.Pipeline(options=unsafe_options) as p:
        _ = (
            p
            | "Create" >> beam.Create([("EMEA", 10.0), ("EMEA", 20.0)])
            | "Window"
            >> beam.WindowInto(
                FixedWindows(size=60),
                trigger=AfterWatermark(),
                accumulation_mode=AccumulationMode.ACCUMULATING,
                allowed_lateness=300,
            )
            | "Group" >> beam.GroupByKey()
        )
except ValueError as exc:
    print("ValueError:", exc)
    print()
    print("Beam refused the combination, which is the correct behaviour.")
    print("The fix is a design decision, not a flag:")
    print("  * set allowed_lateness=0  -> the trigger is safe again")
    print("  * or opt in explicitly and make the sink idempotent")

In [ ]:
# --- The same pipeline with allowed_lateness=0 succeeds ----------------
safe_options = PipelineOptions(["--runner=BundleBasedDirectRunner"])
safe_options.view_as(TypeOptions).allow_unsafe_triggers = False

with beam.Pipeline(options=safe_options) as p:
    _ = (
        p
        | "Create" >> beam.Create([("EMEA", 10.0), ("EMEA", 20.0)])
        | "Window"
        >> beam.WindowInto(
            FixedWindows(size=60),
            trigger=AfterWatermark(),
            accumulation_mode=AccumulationMode.ACCUMULATING,
            allowed_lateness=0,
        )
        | "Group" >> beam.GroupByKey()
    )

print("allowed_lateness=0: the same graph builds and runs cleanly.")
print("The trigger is safe because the window can no longer be updated")
print("after it fires — there is no lateness window to update it in.")

## 8. Practical summary

### What to deploy, and how

| Situation | Deploy | Why |
|---|---|---|
| One-off ETL, rare schedule | a raw `.py` flex template | least ceremony; the pipeline is small and the dependencies are standard |
| Anything that runs daily | a container image | reproducibility; the image pins the whole dependency tree |
| Anything streaming | a container image, always | a mid-flight SDK change in a streaming job is a mid-flight behaviour change |
| Anything writing to BigQuery | a container image | the client library version matters for the write schema |

### The four things to get right, in order

1. **Pin the SDK** with `==`. Ship a container image.
2. **Unique staging and temp** per job, derived from a run-scoped prefix.
3. **The three identities** — caller, service agent, worker SA — with the
   right roles on each.
4. **An idempotent sink**, because jobs get retried.

Everything else in the lifecycle is handled by the control plane.

## 🔑 Takeaways

- The Dataflow job lifecycle (`PENDING` to `RUNNING` to `DRAINING` to `DONE`) is driven by the control plane; `PENDING` is normal, not broken, and a batch job terminates on its own while a streaming job never does.
- A flex template is just a Python file that Dataflow runs on the worker; the old console-registered template model is gone, and the trade-off is that you lose parameter validation in exchange for zero-cost iteration.
- A raw `.py` template resolves the Beam SDK **at launch time on the worker**, which is why an unpinned template can run a different SDK than you tested — pin with `apache-beam[gcp]==2.77.0` and ship a container image.
- `--staging_location` and `--temp_location` are both required for `DataflowRunner` and their defaults collide between concurrent jobs; check them at submission rather than three minutes into `RUNNING`.
- `PipelineOptions` is a union of subclasses: `runner` lives on `StandardOptions` and the cloud flags live on `GoogleCloudOptions`, so `options.view_as(Subclass)` is the only portable way to read them — reading `options.runner` off the base class returns `None` and silently runs the job locally.
- The three identities — the caller, the per-project-region Dataflow service agent, and the worker service account — account for nearly every launch failure; `roles/iam.serviceAccountUser` on the worker account is the one people forget.
- `RuntimeError: A transform with label "Create" already exists in the pipeline` names the offending label, and its suggestion to use `auto_unique_labels` is unsafe for streaming jobs because auto labels change when the graph changes.
- `AfterWatermark` with non-zero allowed lateness raises `ValueError: Unsafe trigger: AfterWatermark() may lose data. Reason: MAY_FINISH` — the fix is a design decision (zero lateness, or lateness plus `ACCUMULATING` plus an idempotent sink), not the `--allow_unsafe_triggers` flag.
- The same pipeline graph running twice produces byte-identical output; reproducing that on Dataflow additionally requires a pinned SDK and a combiner that is insensitive to shard ordering.